# Notebook 01 — DB Setup, Migrations & Auth

**Prerequisites**
1. `docker compose up -d` — Postgres 16 running on localhost:5432
2. Copy `.env.example` → `.env` and verify DATABASE_URL
3. `uv sync` — dependencies installed
4. `uv run alembic upgrade head` — run before this notebook

**What this validates**
- DB connection and table creation
- Seed users + roles
- Password hash + verify
- JWT create + decode
- Expired token rejection
- Union-of-roles access query

In [12]:
import sys
sys.path.insert(0, '..')

import asyncio
import uuid
from datetime import timedelta

from sqlalchemy import select, text
from sqlalchemy.orm import selectinload

from backend.db.session import get_session
from backend.db.models import User, Role, UserRole, Document, DocumentAccess, RoleType, ResponseFormat
from backend.auth.password import hash_password, verify_password
from backend.auth.jwt_handler import create_token, decode_token
from backend.core.exceptions import AuthenticationError

from collections import defaultdict

print('Imports OK')

Imports OK


## 1. DB Connection Test

In [2]:
async def test_connection():
    async with get_session() as session:
        result = await session.execute(text('SELECT version()'))
        version = result.scalar()
        print(f'Connected! Postgres: {version}')

await test_connection()

Connected! Postgres: PostgreSQL 16.12 on aarch64-unknown-linux-musl, compiled by gcc (Alpine 15.2.0) 15.2.0, 64-bit


## 2. Seed Roles

In [3]:
async def seed_roles():
    roles_data = [
        {'role_name': 'system_admin', 'role_type': RoleType.SYSTEM_ADMIN, 'domain': None},
        {'role_name': 'global_auditor', 'role_type': RoleType.GLOBAL_AUDITOR, 'domain': None},
        {'role_name': 'it_auditor', 'role_type': RoleType.DOMAIN_AUDITOR, 'domain': 'IT'},
        {'role_name': 'finance_auditor', 'role_type': RoleType.DOMAIN_AUDITOR, 'domain': 'Finance'},
        {'role_name': 'it_eng', 'role_type': RoleType.FUNCTIONAL, 'domain': 'IT'},
        {'role_name': 'finance', 'role_type': RoleType.FUNCTIONAL, 'domain': 'Finance'},
    ]
    async with get_session() as session:
        for r in roles_data:
            existing = await session.execute(
                select(Role).where(Role.role_name == r['role_name'])
            )
            if existing.scalar_one_or_none() is None:
                role = Role(**r)
                session.add(role)
        result = await session.execute(select(Role))
        roles = result.scalars().all()

    print(f'Roles in DB: {[r.role_name for r in roles]}')
    return roles

roles = await seed_roles()

Roles in DB: ['System Administrators', 'Global Auditors', 'Engineering Auditors', 'Engineering Users', 'system_admin', 'global_auditor', 'it_auditor', 'finance_auditor', 'it_eng', 'finance', 'IT Security Users', 'IT Security Auditors', 'Compliance Users', 'Compliance Auditors', 'Healthcare Users', 'Healthcare Auditors']


## 3. Seed Users

In [6]:
async def seed_users():
    users_data = [
        {'email': 'admin@example.com', 'password': 'adminpassword', 'role': 'system_admin'},
        {'email': 'auditor@example.com', 'password': 'auditpassword', 'role': 'global_auditor'},
        {'email': 'it.user@example.com', 'password': 'itpassword', 'role': 'it_eng'},
        {'email': 'finance.user@example.com', 'password': 'finpassword', 'role': 'finance'},
    ]
    async with get_session() as session:
        for u in users_data:
            existing = await session.execute(select(User).where(User.email == u['email']))
            if existing.scalar_one_or_none() is not None:
                continue
            user = User(
                email=u['email'],
                password_hash=hash_password(u['password']),
                default_format=ResponseFormat.EXECUTIVE_SUMMARY,
            )
            session.add(user)
            await session.flush()  # get user_id
            role_result = await session.execute(select(Role).where(Role.role_name == u['role']))
            role = role_result.scalar_one()
            user_role = UserRole(user_id=user.user_id, role_id=role.role_id)
            session.add(user_role)

        result = await session.execute(select(User))
        users = result.scalars().all()

    print(f'Users in DB: {[u.email for u in users]}')
    
    return users

users = await seed_users()

Users in DB: ['admin@example.com', 'global.auditor@example.com', 'domain.auditor@example.com', 'user@example.com', 'auditor@example.com', 'multi@example.com', 'it.user@example.com', 'finance.user@example.com', 'norole@example.com', 'it.security@example.com', 'compliance@example.com', 'healthcare@example.com']


In [22]:
async def check_all_user_roles():
    """Create a user with 2 roles; verify union access logic."""
    async with get_session() as session:
        # Get roles
        # it_role = (await session.execute(select(Role).where(Role.role_name == 'it_eng'))).scalar_one()
        # fin_role = (await session.execute(select(Role).where(Role.role_name == 'finance'))).scalar_one()

        # # Create multi-role user
        # existing = (await session.execute(select(User).where(User.email == 'multi@example.com'))).scalar_one_or_none()
        # if existing is None:
        #     multi_user = User(email='multi@example.com', password_hash=hash_password('pass'), default_format=ResponseFormat.EXECUTIVE_SUMMARY)
        #     session.add(multi_user)
        #     await session.flush()
        #     session.add(UserRole(user_id=multi_user.user_id, role_id=it_role.role_id))
        #     session.add(UserRole(user_id=multi_user.user_id, role_id=fin_role.role_id))

        # await session.flush()  # flush UserRole inserts (autoflush=False requires explicit flush)

        # Query: what role_ids does this user have?
        result = await session.execute(
            select(User.email, Role.role_name, Role.domain)
            .join(UserRole, UserRole.role_id == Role.role_id)
            .join(User, User.user_id == UserRole.user_id)
        )
        user_roles = result.all()

        grouped_user_roles = defaultdict(list)

        for email, role_name, domain in user_roles:
            grouped_user_roles[email].append(role_name)

    # print(user_roles)
    # print(dict(grouped_user_roles))
    for k, v in grouped_user_roles.items():
        print(f'{k}: {v}')
    # print(f' {[(r.email, r.role_name) for r in user_roles]}')
    # print(f'Multi-role user has roles: {[(r.role_name, r.domain) for r in user_roles]}')
    # assert len(user_roles) == 2, 'Expected 2 roles'
    # domains = {r.domain for r in user_roles}
    # assert 'IT' in domains and 'Finance' in domains
    # print('Union-of-roles access query: PASS')

await check_all_user_roles()

admin@example.com: ['System Administrators']
global.auditor@example.com: ['Global Auditors']
domain.auditor@example.com: ['Engineering Auditors']
user@example.com: ['Engineering Users']
auditor@example.com: ['global_auditor']
multi@example.com: ['it_eng', 'finance']
it.user@example.com: ['it_eng']
finance.user@example.com: ['finance']
it.security@example.com: ['IT Security Users']
compliance@example.com: ['Compliance Users']
healthcare@example.com: ['Healthcare Users']


## 4. Password Hash + Verify

In [5]:
hashed = hash_password('my_secure_password')
print(f'Hash: {hashed[:30]}...')

assert verify_password('my_secure_password', hashed) == True
assert verify_password('wrong_password', hashed) == False
print('Password verification: PASS')

Hash: $2b$12$y2110vPdsmbDlUURBPmxIei...


Password verification: PASS


## 5. JWT Create + Decode

In [6]:
test_user_id = str(uuid.uuid4())
token = create_token(test_user_id, 'test@example.com')
print(f'Token (first 40 chars): {token[:40]}...')

payload = decode_token(token)
assert payload['sub'] == test_user_id
assert payload['email'] == 'test@example.com'
print(f'Decoded payload: sub={payload["sub"][:8]}..., email={payload["email"]}')
print('JWT create + decode: PASS')

Token (first 40 chars): eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJ...
Decoded payload: sub=2900de77..., email=test@example.com
JWT create + decode: PASS


## 6. Expired Token Test

In [7]:
from datetime import datetime, timezone
from jose import jwt
from backend.config import settings

# Manually create an already-expired token
expired_payload = {
    'sub': str(uuid.uuid4()),
    'email': 'expired@example.com',
    'exp': datetime(2020, 1, 1, tzinfo=timezone.utc),  # way in the past
    'iat': datetime(2020, 1, 1, tzinfo=timezone.utc),
}
expired_token = jwt.encode(expired_payload, settings.jwt_secret, algorithm=settings.jwt_algorithm)

try:
    decode_token(expired_token)
    print('ERROR: Should have raised AuthenticationError')
except AuthenticationError as e:
    print(f'Expired token correctly rejected: {e}')
    print('Expired token test: PASS')

Expired token correctly rejected: Invalid or expired token: Signature has expired.
Expired token test: PASS


## 7. Union-of-Roles Access Query

In [8]:
async def test_union_roles():
    """Create a user with 2 roles; verify union access logic."""
    async with get_session() as session:
        # Get roles
        it_role = (await session.execute(select(Role).where(Role.role_name == 'it_eng'))).scalar_one()
        fin_role = (await session.execute(select(Role).where(Role.role_name == 'finance'))).scalar_one()

        # Create multi-role user
        existing = (await session.execute(select(User).where(User.email == 'multi@example.com'))).scalar_one_or_none()
        if existing is None:
            multi_user = User(email='multi@example.com', password_hash=hash_password('pass'), default_format=ResponseFormat.EXECUTIVE_SUMMARY)
            session.add(multi_user)
            await session.flush()
            session.add(UserRole(user_id=multi_user.user_id, role_id=it_role.role_id))
            session.add(UserRole(user_id=multi_user.user_id, role_id=fin_role.role_id))

        await session.flush()  # flush UserRole inserts (autoflush=False requires explicit flush)

        # Query: what role_ids does this user have?
        result = await session.execute(
            select(Role.role_id, Role.role_name, Role.domain)
            .join(UserRole, UserRole.role_id == Role.role_id)
            .join(User, User.user_id == UserRole.user_id)
            .where(User.email == 'multi@example.com')
        )
        user_roles = result.all()
    print(f'Multi-role user has roles: {[(r.role_name, r.domain) for r in user_roles]}')
    assert len(user_roles) == 2, 'Expected 2 roles'
    domains = {r.domain for r in user_roles}
    assert 'IT' in domains and 'Finance' in domains
    print('Union-of-roles access query: PASS')

await test_union_roles()

Multi-role user has roles: [('it_eng', 'IT'), ('finance', 'Finance')]
Union-of-roles access query: PASS


## Summary

All Phase 1 checks passed:
- DB connected
- Roles and users seeded
- Password hashing works (bcrypt)
- JWT create/decode works (HS256)
- Expired tokens are rejected
- Union-of-roles query returns correct role set

**Next:** Run Notebook 02 to validate admin services.